# Prescient PCM Result Analysis

Extract LMP and dispatch data from Prescient simulation results.

Adapted from [RTS-GMLC-PT/Notebook/Prescient_analysis.ipynb](https://github.com/RTS-GMLC-PT) for ERCOT coal retirement case.

**Outputs:**
- `Bus_LMP.csv` — wide format: Datetime × (Bus_LMP, Bus_LMP_DA) for each bus
- `Generator_Dispatch.csv` — wide format: Datetime × (Gen_Dispatch, Gen_Dispatch_DA) for each generator
- `PCM_result.json` — summary statistics (mean/median/min/max) per bus and total dispatch per generator

## 1. Imports & Paths

Set `results_path` to point to your Prescient output directory.

In [ ]:
import os
import json
import numpy as np
import pandas as pd

# === CONFIGURE THIS ===
# Point to the Prescient results directory for the case you want to analyze
results_path = os.path.join(
    os.path.dirname(os.getcwd()),  # adjust as needed
    "data", "retirement_allowed_no_extreme_half_load",
    "Prescient_2", "results"
)

# Output directory (where Bus_LMP.csv, Generator_Dispatch.csv, PCM_result.json will be saved)
output_dir = os.getcwd()

bus_detail_path = os.path.join(results_path, "bus_detail.csv")
thermal_detail_path = os.path.join(results_path, "thermal_detail.csv")
renew_detail_path = os.path.join(results_path, "renewables_detail.csv")

print(f"Results path: {results_path}")
print(f"Output dir:   {output_dir}")
print(f"bus_detail exists: {os.path.exists(bus_detail_path)}")
print(f"thermal_detail exists: {os.path.exists(thermal_detail_path)}")
print(f"renewables_detail exists: {os.path.exists(renew_detail_path)}")

## 2. Helper Functions

Core extraction functions adapted from RTS-GMLC-PT.

In [ ]:
def _prescient_output_to_df(file_name):
    """Load Prescient output CSV and combine Date/Hour/Minute into Datetime."""
    df = pd.read_csv(file_name)
    df['Datetime'] = (
        pd.to_datetime(df['Date'])
        + pd.to_timedelta(df['Hour'], 'hour')
        + pd.to_timedelta(df['Minute'], 'minute')
    )
    df.drop(columns=['Date', 'Hour', 'Minute'], inplace=True)
    # Put Datetime first
    cols = df.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    return df[cols]


def make_lmp_csv(lmp_path, bus_details_path, bus_name, output_dir="."):
    """Extract LMP for a single bus and append to the wide-format LMP CSV.

    Args:
        lmp_path: Path to existing Bus_LMP.csv, or None to create a new one.
        bus_details_path: Path to bus_detail.csv from Prescient results.
        bus_name: Name of the bus to extract.
        output_dir: Directory where Bus_LMP.csv will be saved.
    """
    out_csv = os.path.join(output_dir, "Bus_LMP.csv")
    bdf = _prescient_output_to_df(bus_details_path)
    bdf = bdf[bdf["Bus"] == bus_name][["Datetime", "LMP", "LMP DA"]]
    bdf.set_index("Datetime", inplace=True)
    bdf = bdf.rename(columns={'LMP': f'{bus_name}_LMP', 'LMP DA': f'{bus_name}_LMP DA'})

    if lmp_path is None:
        print(f"Creating Bus_LMP.csv with {bus_name}")
        bdf.to_csv(out_csv)
    else:
        lmp_df = pd.read_csv(lmp_path).set_index("Datetime")
        if f"{bus_name}_LMP" in lmp_df.columns:
            print(f"{bus_name} LMP already exists, skipping.")
            return
        print(f"Adding LMP for {bus_name}")
        bdf_aligned = bdf.reindex(lmp_df.index)
        lmp_df = pd.concat([lmp_df, bdf_aligned], axis=1)
        lmp_df.to_csv(out_csv)


def make_dispatch_csv(dispatch_path, gen_details_path, gen_name, gen_type,
                      other_info=None, output_dir="."):
    """Extract dispatch for a single generator and append to wide-format CSV.

    Args:
        dispatch_path: Path to existing Generator_Dispatch.csv, or None to create new.
        gen_details_path: Path to thermal_detail.csv or renewables_detail.csv.
        gen_name: Generator name/ID.
        gen_type: 'fossil' or 'renew' (determines column names).
        other_info: Additional columns to extract, e.g. ['Curtailment', 'Unit Cost'].
        output_dir: Directory where Generator_Dispatch.csv will be saved.
    """
    out_csv = os.path.join(output_dir, "Generator_Dispatch.csv")
    gdf = _prescient_output_to_df(gen_details_path)

    if gen_type == "fossil":
        info_list = ["Datetime", "Dispatch", "Dispatch DA"]
    elif gen_type == "renew":
        info_list = ["Datetime", "Output", "Output DA"]
    else:
        raise ValueError(f"Unknown gen_type: {gen_type}")

    if other_info is not None:
        info_list.extend(other_info)

    # Convert gen_name to match CSV type (Prescient may store as int or str)
    gdf_filtered = gdf[gdf["Generator"] == gen_name]
    if len(gdf_filtered) == 0:
        # Try numeric conversion
        try:
            gdf_filtered = gdf[gdf["Generator"] == int(gen_name)]
        except (ValueError, TypeError):
            pass
    if len(gdf_filtered) == 0:
        print(f"WARNING: Generator {gen_name} not found in {gen_details_path}")
        return

    gdf_filtered = gdf_filtered[info_list]
    gdf_filtered.set_index("Datetime", inplace=True)

    # Rename columns with generator prefix
    new_col_name = {col: f"{gen_name}_{col}" for col in info_list if col != "Datetime"}
    gdf_filtered = gdf_filtered.rename(columns=new_col_name)

    if dispatch_path is None:
        print(f"Creating Generator_Dispatch.csv with {gen_name}")
        gdf_filtered.to_csv(out_csv)
    else:
        dispatch_df = pd.read_csv(dispatch_path).set_index("Datetime")
        first_col = list(new_col_name.values())[0]
        if first_col in dispatch_df.columns:
            print(f"{gen_name} dispatch already exists, skipping.")
            return
        print(f"Adding dispatch for {gen_name}")
        gdf_aligned = gdf_filtered.reindex(dispatch_df.index)
        dispatch_df = pd.concat([dispatch_df, gdf_aligned], axis=1)
        dispatch_df.to_csv(out_csv)

## 3. Discover Buses and Generators

In [ ]:
# Get unique bus names from bus_detail.csv
bdf_raw = pd.read_csv(bus_detail_path)
bus_names = sorted(bdf_raw["Bus"].unique().tolist())
print(f"Found {len(bus_names)} buses")
print(f"First 10: {bus_names[:10]}")

# Get unique generator names from thermal and renewable details
tdf_raw = pd.read_csv(thermal_detail_path)
thermal_gens = sorted(tdf_raw["Generator"].unique().tolist())
print(f"\nFound {len(thermal_gens)} thermal generators")

rdf_raw = pd.read_csv(renew_detail_path)
renew_gens = sorted(rdf_raw["Generator"].unique().tolist())
print(f"Found {len(renew_gens)} renewable generators")

## 4. Extract LMP for All Buses

In [ ]:
lmp_csv = os.path.join(output_dir, "Bus_LMP.csv")

for idx, bus_name in enumerate(bus_names):
    if idx == 0:
        make_lmp_csv(lmp_path=None, bus_details_path=bus_detail_path,
                     bus_name=bus_name, output_dir=output_dir)
    else:
        make_lmp_csv(lmp_path=lmp_csv, bus_details_path=bus_detail_path,
                     bus_name=bus_name, output_dir=output_dir)

# Verify
df_lmp_check = pd.read_csv(lmp_csv)
print(f"\nBus_LMP.csv: {df_lmp_check.shape[0]} rows × {df_lmp_check.shape[1]} columns")
print(f"Expected: {len(bus_names)*2 + 1} columns (Datetime + 2 per bus)")

## 5. Extract Dispatch for All Generators

In [ ]:
dispatch_csv = os.path.join(output_dir, "Generator_Dispatch.csv")

# Thermal generators
thermal_other_info = ["Unit Cost", "Unit State"]
for idx, gen_name in enumerate(thermal_gens):
    if idx == 0:
        make_dispatch_csv(dispatch_path=None, gen_details_path=thermal_detail_path,
                          gen_name=gen_name, gen_type="fossil",
                          other_info=thermal_other_info, output_dir=output_dir)
    else:
        make_dispatch_csv(dispatch_path=dispatch_csv, gen_details_path=thermal_detail_path,
                          gen_name=gen_name, gen_type="fossil",
                          other_info=thermal_other_info, output_dir=output_dir)

# Renewable generators
renew_other_info = ["Curtailment"]
for gen_name in renew_gens:
    make_dispatch_csv(dispatch_path=dispatch_csv, gen_details_path=renew_detail_path,
                      gen_name=gen_name, gen_type="renew",
                      other_info=renew_other_info, output_dir=output_dir)

# Verify
df_disp_check = pd.read_csv(dispatch_csv)
print(f"\nGenerator_Dispatch.csv: {df_disp_check.shape[0]} rows × {df_disp_check.shape[1]} columns")

## 6. Summary Statistics

In [ ]:
df_lmp = pd.read_csv(lmp_csv)
df_dispatch = pd.read_csv(dispatch_csv)

# LMP summary per bus
LMP_result = {}
for bus_name in bus_names:
    lmp_col = f"{bus_name}_LMP"
    lmp_da_col = f"{bus_name}_LMP DA"
    if lmp_da_col not in df_lmp.columns:
        print(f"WARNING: {lmp_da_col} not found in Bus_LMP.csv")
        continue
    LMP_result[bus_name] = {
        "LMP_DA_mean": df_lmp[lmp_da_col].mean(),
        "LMP_DA_median": df_lmp[lmp_da_col].median(),
        "LMP_DA_min": df_lmp[lmp_da_col].min(),
        "LMP_DA_max": df_lmp[lmp_da_col].max(),
        "LMP_mean": df_lmp[lmp_col].mean(),
        "LMP_median": df_lmp[lmp_col].median(),
        "LMP_min": df_lmp[lmp_col].min(),
        "LMP_max": df_lmp[lmp_col].max(),
    }

# Dispatch summary per generator
dispatch_result = {}

for gen_name in thermal_gens:
    gen_str = str(gen_name)
    da_col = f"{gen_str}_Dispatch DA"
    rt_col = f"{gen_str}_Dispatch"
    if da_col in df_dispatch.columns:
        dispatch_result[gen_str] = {
            "type": "thermal",
            "tot_Dispatch_DA": float(df_dispatch[da_col].sum()),
            "tot_Dispatch": float(df_dispatch[rt_col].sum()),
        }

for gen_name in renew_gens:
    gen_str = str(gen_name)
    da_col = f"{gen_str}_Output DA"
    rt_col = f"{gen_str}_Output"
    if da_col in df_dispatch.columns:
        entry = {
            "type": "renewable",
            "tot_Output_DA": float(df_dispatch[da_col].sum()),
            "tot_Output": float(df_dispatch[rt_col].sum()),
        }
        curt_col = f"{gen_str}_Curtailment"
        if curt_col in df_dispatch.columns:
            entry["tot_Curtailment"] = float(df_dispatch[curt_col].sum())
        dispatch_result[gen_str] = entry

print(f"LMP stats for {len(LMP_result)} buses")
print(f"Dispatch stats for {len(dispatch_result)} generators")

## 7. Save Summary JSON

In [ ]:
result_summary = {
    "LMP": LMP_result,
    "Dispatch": dispatch_result,
}

pcm_result_path = os.path.join(output_dir, "PCM_result.json")
with open(pcm_result_path, "w") as f:
    json.dump(result_summary, f, indent=2)

print(f"Saved to {pcm_result_path}")

## 8. Quick Sanity Check

In [ ]:
import matplotlib.pyplot as plt

# Pick a few buses to plot
sample_buses = bus_names[:5]

fig, axes = plt.subplots(len(sample_buses), 1, figsize=(14, 3 * len(sample_buses)), sharex=True)
df_lmp["Datetime"] = pd.to_datetime(df_lmp["Datetime"])

for ax, bus_name in zip(axes, sample_buses):
    ax.plot(df_lmp["Datetime"], df_lmp[f"{bus_name}_LMP DA"], label="LMP DA", alpha=0.7)
    ax.plot(df_lmp["Datetime"], df_lmp[f"{bus_name}_LMP"], label="LMP RT", alpha=0.5)
    ax.set_ylabel("$/MWh")
    ax.set_title(bus_name)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
# LMP distribution across all buses
lmp_da_means = [LMP_result[b]["LMP_DA_mean"] for b in bus_names if b in LMP_result]
lmp_da_buses = [b for b in bus_names if b in LMP_result]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(lmp_da_means)), lmp_da_means)
ax.set_xticks(range(len(lmp_da_buses)))
ax.set_xticklabels(lmp_da_buses, rotation=90, fontsize=6)
ax.set_ylabel("Mean DA LMP ($/MWh)")
ax.set_title("Mean Day-Ahead LMP by Bus")
plt.tight_layout()
plt.show()